# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [1]:
#1 Imports and Warnings Suppression

In [2]:
import requests
from bs4 import BeautifulSoup
from newspaper import Article
from youtube_transcript_api import YouTubeTranscriptApi
from itext2kg.documents_distiller import DocumentsDistiller, CV
import yaml
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.caches import BaseCache
from pydantic import BaseModel
from langchain_core.callbacks.base import Callbacks
from typing import List
from itext2kg import iText2KG
import networkx as nx
import matplotlib.pyplot as plt
from langchain.text_splitter import RecursiveCharacterTextSplitter
import time
import warnings

# Suppress Pydantic schema warning
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic._internal._generate_schema")

ChatOpenAI.BaseCache = BaseCache
ChatOpenAI.model_rebuild()


C:\Program Files\Python310\lib\site-packages\pydantic\_internal\_generate_schema.py:628: UserWarning: <built-in function array> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


True

In [3]:
#2 Article Scraping Function 

In [4]:
def scrape_article(url):
    print(f"Scraping article from: {url}")
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/114.0.0.0 Safari/537.36"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        
        # Method 1: BeautifulSoup approach
        soup = BeautifulSoup(response.text, 'html.parser')
        content_div = soup.find("div", class_="entry-content ap-pattern--entry-content")
        
        if content_div:
            article_text = content_div.get_text(separator="\n", strip=True)
            with open("pure_scrum_clean.txt", "w", encoding="utf-8") as f:
                f.write(article_text)
            print("Article scraped successfully with BeautifulSoup")
        else:
            print("Specific div not found, trying newspaper3k...")
            
        # Method 2: Newspaper3k approach (as backup)
        article = Article(url)
        article.download()
        article.parse()
        
        if article.text:
            with open("pure_scrum_clean2.txt", "w", encoding="utf-8") as f:
                f.write(article.text)
            print("Article scraped successfully with newspaper3k")
            return article.text
        else:
            print("Failed to extract article content")
            return None
            
    except requests.RequestException as e:
        print(f"Error fetching article: {e}")
        return None

In [5]:
#3. YouTube Transcript Retrieval Function

In [6]:
def get_youtube_transcript(video_id):
    print(f"Getting YouTube transcript for video: {video_id}")
    
    try:
        transcript_list = YouTubeTranscriptApi.get_transcript(video_id)
        transcript = " ".join([item['text'] for item in transcript_list])
        
        with open("youtubeVideo.txt", "w", encoding="utf-8") as f:
            f.write(transcript)
        
        print("YouTube transcript downloaded successfully")
        return transcript
        
    except Exception as e:
        print(f"Error getting YouTube transcript: {e}")
        return None

In [7]:
#4. Configuration Loading

In [8]:
def load_config(config_path="config.yml"):
    try:
        with open(config_path, "r") as f:
            config = yaml.safe_load(f)
        return config
    except FileNotFoundError:
        print("config.yml not found. Creating sample config...")
        sample_config = {
            'llm': {
                'model_type': 'openai',
                'model_name': 'gpt-3.5-turbo',
                'temperature': 0.7,
                'max_tokens': 2000,
                'api_key': '',
                'timeout': 60,
                'max_retries': 3
            },
            'paths': {
                'output_dir': './output'
            }
        }
        with open(config_path, "w") as f:
            yaml.dump(sample_config, f, default_flow_style=False)
        print("Sample config.yml created. Please add your OpenAI API key.")
        return None

In [9]:
#5. LLM and Embeddings Initialization

In [10]:
import os


def initialize_llm(config):
    if not config:
        return None, None
    
    llm_config = config.get('llm', {})
    api_key = llm_config.get('api_key')
    
    if not api_key:
        api_key = os.getenv('OPENAI_API_KEY')
        if not api_key:
            print("OpenAI API key not found. Please set OPENAI_API_KEY environment variable or update config.yml")
            return None, None
    
    try:
        if llm_config.get('model_type') == "openai":
            llm = ChatOpenAI(
                model=llm_config.get('model_name', 'gpt-3.5-turbo'),
                temperature=llm_config.get('temperature', 0.7),
                max_tokens=llm_config.get('max_tokens', 2000),
                api_key=api_key,
                request_timeout=llm_config.get('timeout', 60),
                max_retries=llm_config.get('max_retries', 3)
            )
            embeddings = OpenAIEmbeddings(api_key=api_key)
            print("LLM and Embeddings initialized successfully")
            return llm, embeddings
        else:
            print("Only 'openai' model type is supported")
            return None, None
    except Exception as e:
        print(f"Error initializing LLM: {e}")
        return None, None

In [11]:
#6. Pydantic Models for Triples

In [12]:
class Triple(BaseModel):
    subject: str
    predicate: str
    object: str

class TriplesOutput(BaseModel):
    triples: List[Triple]


In [13]:
#7. Knowledge Graph Extraction Function with Retry

In [14]:
def simple_kg_extraction(text, llm, max_retries=3):
    if not llm:
        print("LLM not initialized")
        return []
    
    prompt = f"""
    Extract knowledge graph triples from the following text in the format (subject, predicate, object).
    Focus on the most important relationships and entities.
    
    Text: {text[:1500]}...
    
    Please return triples in the format:
    - (Entity1, relationship, Entity2)
    - (Entity1, relationship, Entity2)
    
    Limit to the 8 most important triples.
    """
    
    for attempt in range(max_retries):
        try:
            response = llm.invoke(prompt)
            lines = response.content.split('\n')
            triples = []
            for line in lines:
                if line.strip().startswith('-') and '(' in line and ')' in line:
                    start = line.find('(')
                    end = line.find(')')
                    if start != -1 and end != -1:
                        triple_str = line[start+1:end]
                        parts = [part.strip() for part in triple_str.split(',')]
                        if len(parts) == 3:
                            triples.append((parts[0], parts[1], parts[2]))
            
            return triples
            
        except Exception as e:
            error_msg = str(e).lower()
            if "rate limit" in error_msg or "429" in error_msg:
                wait_time = (2 ** attempt) * 10
                print(f"Rate limit hit (attempt {attempt + 1}/{max_retries}). Waiting {wait_time}s...")
                time.sleep(wait_time)
            elif "token" in error_msg:
                print(f"Token limit exceeded. Reducing text size...")
                text = text[:len(text)//2]
                prompt = f"""
                Extract knowledge graph triples from the following text in the format (subject, predicate, object).
                Focus on the most important relationships and entities.
                
                Text: {text}
                
                Please return triples in the format:
                - (Entity1, relationship, Entity2)
                - (Entity1, relationship, Entity2)
                
                Limit to the 5 most important triples.
                """
                time.sleep(5)
            else:
                print(f"Error in KG extraction (attempt {attempt + 1}/{max_retries}): {e}")
                time.sleep(5)
    
    print(f"Failed to extract triples after {max_retries} attempts")
    return []


In [15]:
#8. Process Text in Chunks to Avoid Rate Limits

In [16]:
def process_text_in_chunks(text, llm, chunk_size=3000, chunk_overlap=100):
    print(f"Processing text in chunks...")
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_text(text)
    
    print(f"Created {len(chunks)} chunks")
    
    if len(chunks) > 10:
        print(f"Too many chunks ({len(chunks)}). Processing only first 10 to avoid rate limits.")
        chunks = chunks[:10]
    
    all_triples = []
    DELAY_BETWEEN_CHUNKS = 15
    
    for i, chunk in enumerate(chunks, 1):
        print(f"\nProcessing chunk {i}/{len(chunks)}")
        
        triples = simple_kg_extraction(chunk, llm)
        if triples:
            all_triples.extend(triples)
            print(f"Chunk {i} → {len(triples)} triples extracted")
        else:
            print(f"Chunk {i} → No triples extracted")
        
        if i < len(chunks):
            print(f"Waiting {DELAY_BETWEEN_CHUNKS}s before next chunk to avoid rate limits...")
            time.sleep(DELAY_BETWEEN_CHUNKS)
    
    return all_triples


In [17]:
#8. Save Triples to File

In [18]:
def save_triples(triples, filename="extracted_triples.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        f.write("Extracted Knowledge Graph Triples:\n")
        f.write("=" * 40 + "\n\n")
        for i, (subject, predicate, obj) in enumerate(triples, 1):
            f.write(f"{i}. ({subject}) --[{predicate}]--> ({obj})\n")
    
    print(f"Triples saved to {filename}")


In [19]:
#9. Main Execution Logic

In [20]:
def main():
    print("Starting Knowledge Graph Extraction Pipeline\n")
    
    # Scrape article
    url = "https://age-of-product.com/pure-scrum/"
    article_text = scrape_article(url)
    
    # Get YouTube transcript
    video_id = "502ILHjX9EE"
    youtube_text = get_youtube_transcript(video_id)
    
    # Choose which text to process
    if youtube_text:
        input_text = youtube_text
        print("Using YouTube transcript for KG extraction")
    elif article_text:
        input_text = article_text
        print("Using article text for KG extraction")
    else:
        print("No text available for processing")
        return
    
    # Load configuration and initialize LLM
    config = load_config()
    llm, embeddings = initialize_llm(config)
    
    if not llm:
        print("Cannot proceed without LLM initialization")
        return
    
    # Process text and extract knowledge graph
    print(f"\nInput text length: {len(input_text)} characters")
    all_triples = process_text_in_chunks(input_text, llm)
    
    # Save results
    if all_triples:
        print(f"\nTotal triples extracted: {len(all_triples)}")
        save_triples(all_triples)
        
        print("\nSample triples:")
        for i, (subject, predicate, obj) in enumerate(all_triples[:5], 1):
            print(f"  {i}. ({subject}) --[{predicate}]--> ({obj})")
        
        if len(all_triples) > 5:
            print(f"  ... and {len(all_triples) - 5} more triples")
    else:
        print("No triples were successfully extracted")


In [21]:
#10. Run the main() function

In [22]:
if __name__ == "__main__":
    main()


Starting Knowledge Graph Extraction Pipeline

Scraping article from: https://age-of-product.com/pure-scrum/
Article scraped successfully with BeautifulSoup
Article scraped successfully with newspaper3k
Getting YouTube transcript for video: 502ILHjX9EE
YouTube transcript downloaded successfully
Using YouTube transcript for KG extraction
LLM and Embeddings initialized successfully

Input text length: 17452 characters
Processing text in chunks...
Created 8 chunks

Processing chunk 1/8
Rate limit hit (attempt 1/3). Waiting 10s...
Rate limit hit (attempt 2/3). Waiting 20s...
Rate limit hit (attempt 3/3). Waiting 40s...
Failed to extract triples after 3 attempts
Chunk 1 → No triples extracted
Waiting 15s before next chunk to avoid rate limits...

Processing chunk 2/8
Rate limit hit (attempt 1/3). Waiting 10s...
Rate limit hit (attempt 2/3). Waiting 20s...
Rate limit hit (attempt 3/3). Waiting 40s...
Failed to extract triples after 3 attempts
Chunk 2 → No triples extracted
Waiting 15s before 